In [21]:
def get_skewed_p(n, skew=5, loc=10, scale=5):
    p = skewnorm.pdf(np.arange(n), a=skew, loc=loc, scale=scale)
    return p / p.sum()

def get_powerlaw_p(n, a=2.0):
    p = (np.arange(1, n + 1).astype(float)) ** -a
    return p / p.sum()

def get_quasi_constant_p(n, dominance=0.9):
    p = np.full(n, (1 - dominance) / (n - 1))
    p[0] = dominance  # Make the first item dominant
    return p

def get_bimodal_p(n):
    x = np.linspace(-3, 3, n)
    # Combine two Gaussian curves centered at different points
    p = np.exp(-(x-1.5)**2) + np.exp(-(x+1.5)**2)
    return p / p.sum()

In [22]:
"""

Generated using Claude Sonnet 4.6 - 3rd March 2026

seed_to_csv.py
────────────────────────────────────────────────────────────────────────────────
Generates fake research data for all tables and writes:
  • one CSV per table into an output directory (./csv_output/ by default)
  • cs_db.json  – all CS + shared tables in json-server format

Requirements:
    pip install faker

Usage:
    python seed_to_csv.py                  # writes to ./csv_output/
    python seed_to_csv.py --out /tmp/data  # custom directory

json-server quickstart:
    npx json-server --watch csv_output/cs_db.json --port 3001
    # GET /cs_algorithm  GET /cs_benchmark  GET /cs_experiment_run  …
"""

import csv
import json
import random
from datetime import date, timedelta, datetime
from pathlib import Path
import numpy as np
from scipy.stats import skewnorm

from faker import Faker

# ── CLI ───────────────────────────────────────────────────────────────────────


OUT = Path("csv_output")
OUT.mkdir(parents=True, exist_ok=True)

fake = Faker()
Faker.seed(3467890345)
random.seed(8906773862)

# ── Helpers ───────────────────────────────────────────────────────────────────

def rnd_date(start: date, end: date) -> date:
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, max(delta, 0)))

def rnd_score(lo=0.0, hi=1.0) -> float:
    return round(random.uniform(lo, hi), 4)

def wc(options, weights=None):
    return random.choices(options, weights=weights, k=1)[0]

_id_counters: dict[str, int] = {}
def next_id(table: str) -> int:
    _id_counters[table] = _id_counters.get(table, 0) + 1
    return _id_counters[table]

def write_csv(name: str, rows: list[dict]) -> None:
    if not rows:
        print(f"  {name:<40} 0 rows  (skipped)")
        return
    path = OUT / f"{name}.csv"
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"  {name:<40} {len(rows):>6} rows  →  {path}")

# ── Reference data ────────────────────────────────────────────────────────────

AGE_BRACKETS       = ["18-24","25-34","35-44","45-54","55-64","65+"]
GENDERS            = ["Male","Female","Non-binary","Prefer not to say"]
EDUCATION          = ["High School","Some College","Bachelor's","Master's","PhD","Other"]
EMP_STATUSES       = ["Full-time","Part-time","Self-employed","Unemployed","Student","Retired"]
INCOME_BRACKETS    = ["<30k","30-60k","60-100k","100k+"]
COUNTRIES          = ["US","DE","GB","FR","CA","AU","NL","SE","JP","BR"]
SCALE_TYPES        = ["Likert-5","Likert-7","VAS","Binary","Semantic-Diff"]
INST_DOMAINS       = ["Consumer Psych","UX","Cognitive Load","Financial Anxiety","Brand Trust","Job Satisfaction"]
DS_DOMAINS         = ["Economics","Behavioural","NLP","Computer Vision","Healthcare","Social Science","Finance"]
LICENSES           = ["CC-BY-4.0","MIT","Apache-2.0","ODC-By","Public Domain"]
ALGO_FAMILIES      = ["Gradient Boosting","Transformer","SVM","Neural Network","Random Forest","Logistic Regression","KNN"]
TASK_TYPES         = ["Classification","Regression","Clustering","NLP","Recommendation"]
EVAL_METRICS       = ["F1","RMSE","AUC","Accuracy","MAE","BLEU","Precision","Recall"]
HARDWARE_TAGS      = ["A100-40GB","V100-32GB","RTX-4090","M2-Pro-32GB","CPU-only-64GB","TPU-v3"]
RESEARCH_TYPES     = ["Cross-sectional Survey","Longitudinal Panel","Lab Experiment","Field Study","Mixed Methods"]
STUDY_TOPICS       = [
    "Employment Status → Consumer Spending",
    "Income Shock → Financial Decision Reversal",
    "Job Security → Brand Loyalty",
    "Gig Economy → Subscription Behaviour",
    "Unemployment Duration → Savings Rate",
    "Remote Work → Discretionary Spending",
    "Income Inequality → Market Confidence",
    "Part-time Work → Credit Card Usage",
]
DECISION_TYPES     = ["Major Purchase","Subscription","Savings Deposit","Credit Application","Investment","Subscription Cancel","Loan Repayment"]
JOB_SECTORS        = ["Technology","Healthcare","Finance","Retail","Education","Manufacturing","Public Sector","Hospitality","Construction"]
USAGE_ROLES        = ["Baseline","Validation","Supplementary","Primary Dataset","Comparative"]
PROTOCOLS          = ["Within-subject","Between-subject","Mixed","Counterbalanced"]
TASK_LABELS        = [
    "Dashboard navigation","Search & filter","Recommendation acceptance",
    "Anomaly identification","Report export","Model output interpretation",
    "Alert triage","Onboarding flow","Checkout completion",
]
METRICS_BY_TASK = {
    "Classification":  ["Accuracy","F1","AUC","Precision","Recall"],
    "Regression":      ["RMSE","MAE","R2"],
    "Clustering":      ["Silhouette","Davies-Bouldin"],
    "NLP":             ["BLEU","F1","Accuracy"],
    "Recommendation":  ["RMSE","Precision","Recall"],
}
COMMON_HPS = {
    "Gradient Boosting":   [("n_estimators","int"),("learning_rate","float"),("max_depth","int"),("subsample","float")],
    "Transformer":         [("num_layers","int"),("hidden_size","int"),("num_heads","int"),("dropout","float"),("lr","float")],
    "SVM":                 [("C","float"),("kernel","str"),("gamma","str")],
    "Neural Network":      [("layers","int"),("lr","float"),("batch_size","int"),("epochs","int"),("dropout","float")],
    "Random Forest":       [("n_estimators","int"),("max_depth","int"),("min_samples_split","int")],
    "Logistic Regression": [("C","float"),("max_iter","int"),("solver","str")],
    "KNN":                 [("n_neighbors","int"),("metric","str"),("weights","str")],
}
HP_VALUES = {
    "int":   lambda: str(random.choice([8,16,32,64,100,128,200,256,512])),
    "float": lambda: str(round(random.uniform(0.0001, 1.0), 5)),
    "str":   lambda: random.choice(["auto","rbf","linear","adam","sgd","cosine","euclidean"]),
}

# ═══════════════════════════════════════════════════════════════════════════════
#  SHARED TABLES
# ═══════════════════════════════════════════════════════════════════════════════

print("\nShared tables")

# ── survey_instrument ──────────────────────────────────────────────────────────
INSTRUMENT_NAMES = [
    "Likert-7 Financial Anxiety Scale",
    "VAS Consumer Confidence Index",
    "Likert-5 Brand Trust Scale",
    "Binary Employment Hardship Screen",
    "Semantic-Diff Job Satisfaction",
    "Likert-7 UX Satisfaction (SUS-derived)",
    "NASA-TLX Cognitive Load Instrument",
    "VAS Perceived Effort Scale",
    "Likert-5 Technology Acceptance (TAM)",
    "Likert-7 Impulse Buying Tendency",
    "Binary Financial Literacy Screener",
    "Likert-5 Price Sensitivity Scale",
    "Semantic-Diff Brand Perception",
    "Likert-7 Work Engagement Scale",
    "VAS Stress & Burnout Indicator",
]
instruments = []
for name in INSTRUMENT_NAMES:
    instruments.append({
        "instrument_id": next_id("instrument"),
        "name":          name,
        "scale_type":    random.choice(SCALE_TYPES),
        "item_count":    random.randint(5, 24),
        "validated":     random.random() > 0.3,
        "language":      "en",
        "domain":        random.choice(INST_DOMAINS),
    })
write_csv("survey_instrument", instruments)
instrument_ids = [r["instrument_id"] for r in instruments]

# ── open_dataset ───────────────────────────────────────────────────────────────
DATASET_NAMES = [
    ("BLS Current Population Survey",          "Economics"),
    ("UCI Adult Income Dataset",                "Social Science"),
    ("Kaggle Consumer Behaviour Survey 2023",   "Behavioural"),
    ("Federal Reserve Consumer Finance Panel",  "Finance"),
    ("OECD Employment Outlook Microdata",       "Economics"),
    ("Nielsen Homescan Panel Extract",          "Behavioural"),
    ("Twitter Financial Sentiment Corpus",      "NLP"),
    ("Amazon Product Reviews (5-star subset)",  "NLP"),
    ("MNIST Handwritten Digits",                "Computer Vision"),
    ("ImageNet-1k Validation Set",              "Computer Vision"),
    ("UCI Heart Disease Dataset",               "Healthcare"),
    ("MIMIC-III Clinical Notes (de-id)",        "Healthcare"),
    ("World Bank WDI Time Series",              "Economics"),
    ("Stack Overflow Developer Survey 2023",    "Social Science"),
    ("GitHub Activity Graph Dataset",           "Social Science"),
    ("MovieLens 25M Ratings",                   "Recommendation"),
    ("Open Payments CMS Drug Data",             "Healthcare"),
    ("EuroStat Labour Force Survey",            "Economics"),
    ("Yelp Open Dataset (Business)",            "Behavioural"),
    ("Common Crawl News Corpus (en)",           "NLP"),
    ("Pew Research Political Survey 2022",      "Social Science"),
    ("Quandl NASDAQ Price History",             "Finance"),
    ("UK Biobank Lifestyle Questionnaire",      "Healthcare"),
    ("Behavioural Risk Factor Surveillance",    "Behavioural"),
    ("MS COCO Object Detection 2017",           "Computer Vision"),
    ("UCI Bank Marketing Dataset",              "Finance"),
    ("GoodReads Book Reviews",                  "NLP"),
    ("OpenStreetMap POI Density Grid",          "Social Science"),
    ("European Social Survey Round 10",         "Social Science"),
    ("GlobalJobsMiner LinkedIn Scrape",         "Economics"),
]
datasets = []
for dname, ddomain in DATASET_NAMES:
    datasets.append({
        "dataset_id":     next_id("dataset"),
        "name":           dname,
        "source_url":     fake.url(),
        "domain":         ddomain,
        "row_count":      random.randint(5_000, 10_000_000),
        "feature_count":  random.randint(8, 256),
        "license":        random.choice(LICENSES),
        "published_year": random.randint(2015, 2023),
    })
write_csv("open_dataset", datasets)
dataset_ids = [r["dataset_id"] for r in datasets]
econ_dataset_ids = [r["dataset_id"] for r in datasets
                    if r["domain"] in ("Economics","Behavioural","Finance","Social Science")]

# ── respondent ─────────────────────────────────────────────────────────────────
N_RESPONDENTS = 800
respondents = []
for _ in range(N_RESPONDENTS):
    emp = wc(EMP_STATUSES, [35,15,10,15,15,10])
    inc = wc(INCOME_BRACKETS,
             [30,35,25,10] if emp in ("Unemployed","Student") else [10,30,40,20])
    respondents.append({
        "respondent_id":    next_id("respondent"),
        "age_bracket":      random.choice(AGE_BRACKETS),
        "gender":           random.choice(GENDERS),
        "education_level":  random.choice(EDUCATION),
        "employment_status":emp,
        "country_iso":      random.choice(COUNTRIES),
        "income_bracket":   inc,
        "consent_given":    True,
        "recruited_at":     rnd_date(date(2020,1,1), date(2024,6,1)),
    })
write_csv("respondent", respondents)
respondent_ids  = [r["respondent_id"]    for r in respondents]
respondent_meta = {r["respondent_id"]: r for r in respondents}

# ═══════════════════════════════════════════════════════════════════════════════
#  BUSINESS DEPARTMENT
# ═══════════════════════════════════════════════════════════════════════════════

print("\nBusiness tables")

# ── biz_study ──────────────────────────────────────────────────────────────────
biz_studies = []
for _ in range(20):
    start = rnd_date(date(2019,1,1), date(2023,6,1))
    end   = rnd_date(start + timedelta(days=60), start + timedelta(days=540))
    target_n = random.choice([150,200,300,400,500,800,1000])
    biz_studies.append({
        "study_id":      next_id("biz_study"),
        "title":         fake.catch_phrase() + " — " + random.choice(STUDY_TOPICS),
        "research_type": random.choice(RESEARCH_TYPES),
        "topic":         random.choice(STUDY_TOPICS),
        "start_date":    start,
        "end_date":      end,
        "target_n":      target_n,
        "achieved_n":    int(target_n * random.uniform(0.70, 1.00)),
        "status":        wc(["Active","Completed","Paused"], [10,80,10]),
    })
write_csv("biz_study", biz_studies)
biz_study_ids   = [r["study_id"]   for r in biz_studies]
biz_study_dates = {r["study_id"]: (r["start_date"], r["end_date"]) for r in biz_studies}

# ── biz_study_dataset ──────────────────────────────────────────────────────────
biz_study_datasets = []
biz_study_econ_ds  = {}
seen_pairs = set()
for sid in biz_study_ids:
    chosen = random.sample(econ_dataset_ids, k=random.randint(1, 3))
    biz_study_econ_ds[sid] = chosen
    for did in chosen:
        if (sid, did) not in seen_pairs:
            seen_pairs.add((sid, did))
            biz_study_datasets.append({
                "study_id":    sid,
                "dataset_id":  did,
                "usage_role":  random.choice(USAGE_ROLES),
                "rows_sampled":random.randint(500, 500_000),
            })
write_csv("biz_study_dataset", biz_study_datasets)

# ── biz_survey_response ────────────────────────────────────────────────────────
biz_responses = []
for sid in biz_study_ids:
    start, end = biz_study_dates[sid]
    pool = random.sample(respondent_ids, k=random.randint(40, 80))
    for rid in pool:
        biz_responses.append({
            "response_id":    next_id("biz_response"),
            "study_id":       sid,
            "respondent_id":  rid,
            "instrument_id":  random.choice(instrument_ids),
            "collected_at":   datetime.combine(rnd_date(start, end), datetime.min.time()),
            "channel":        random.choice(["Online","Phone","In-person"]),
            "completion_secs":random.randint(120, 1800),
            "is_complete":    random.random() > 0.12,
        })
write_csv("biz_survey_response", biz_responses)

# ── biz_response_item ──────────────────────────────────────────────────────────
biz_items = []
for resp in biz_responses:
    if resp["is_complete"]:
        for idx in range(1, random.randint(6, 21)):
            biz_items.append({
                "response_id":  resp["response_id"],
                "item_index":   idx,
                "raw_answer":   str(random.randint(1, 7)),
                "numeric_score":round(random.uniform(0, 1), 4),
            })
write_csv("biz_response_item", biz_items)

# ── biz_consumer_decision ──────────────────────────────────────────────────────
biz_decisions = []
for sid in biz_study_ids:
    start, end = biz_study_dates[sid]
    for _ in range(random.randint(60, 100)):
        rid  = random.choice(respondent_ids)
        meta = respondent_meta[rid]
        emp  = meta["employment_status"]
        base = 5000 if emp == "Full-time" else 1500 if emp in ("Part-time","Self-employed") else 400
        biz_decisions.append({
            "decision_id":      next_id("biz_decision"),
            "respondent_id":    rid,
            "study_id":         sid,
            "decision_type":    random.choice(DECISION_TYPES),
            "employment_status":emp,
            "income_at_time":   meta["income_bracket"],
            "amount_usd":       round(random.uniform(base * 0.1, base * 3.5), 2),
            "decision_date":    rnd_date(start, end),
            "reversed":         random.random() < (0.05 if emp == "Full-time" else 0.20),
            "confidence_score": round(random.uniform(0.3, 1.0) if emp == "Full-time"
                                      else random.uniform(0.1, 0.75), 3),
        })
write_csv("biz_consumer_decision", biz_decisions)

# ── biz_employment_wave ────────────────────────────────────────────────────────
biz_waves = []
for sid in biz_study_ids:
    start, end = biz_study_dates[sid]
    panel = random.sample(respondent_ids, k=min(60, len(respondent_ids)))
    n_waves = random.randint(2, 4)
    wave_dates = sorted(rnd_date(start, end) for _ in range(n_waves))
    for rid in panel:
        emp = respondent_meta[rid]["employment_status"]
        for wd in wave_dates:
            if random.random() < 0.08:
                emp = random.choice(EMP_STATUSES)
            biz_waves.append({
                "wave_id":          next_id("biz_wave"),
                "study_id":         sid,
                "respondent_id":    rid,
                "wave_date":        wd,
                "employment_status":emp,
                "job_sector":       random.choice(JOB_SECTORS),
                "weekly_hours":     round(random.uniform(0, 50), 2) if emp not in ("Unemployed","Retired") else 0,
                "gross_monthly_usd":round(random.uniform(1500, 12000), 2) if emp == "Full-time"
                                    else round(random.uniform(500, 4000), 2) if emp in ("Part-time","Self-employed","Student")
                                    else round(random.uniform(0, 1200), 2),
                "job_satisfaction": rnd_score(),
                "financial_anxiety":rnd_score(),
            })
write_csv("biz_employment_wave", biz_waves)

# ── biz_market_sentiment ───────────────────────────────────────────────────────
SEGMENTS = [
    f"{emp} {age} {cc}"
    for emp in ["Full-time","Unemployed","Part-time"]
    for age in ["18-34","35-54","55+"]
    for cc in ["US","DE","GB"]
]
biz_sentiments = []
for sid in biz_study_ids:
    start, end = biz_study_dates[sid]
    chosen_segs = random.sample(SEGMENTS, k=random.randint(4, 10))
    period = start
    while period < end:
        p_end = min(period + timedelta(days=90), end)
        seg   = random.choice(chosen_segs)
        unmp  = "Unemployed" in seg
        biz_sentiments.append({
            "sentiment_id":   next_id("biz_sentiment"),
            "study_id":       sid,
            "segment_label":  seg,
            "period_start":   period,
            "period_end":     p_end,
            "n_respondents":  random.randint(20, 180),
            "avg_confidence": round(random.uniform(0.3, 0.85) if not unmp else random.uniform(0.1, 0.5), 4),
            "avg_spend_intent":round(random.uniform(0.25, 0.80) if not unmp else random.uniform(0.05, 0.35), 4),
            "avg_save_intent": round(random.uniform(0.15, 0.60) if not unmp else random.uniform(0.25, 0.75), 4),
            "dataset_id":     random.choice(biz_study_econ_ds.get(sid, [None])),
        })
        period = p_end + timedelta(days=1)
write_csv("biz_market_sentiment", biz_sentiments)

# ═══════════════════════════════════════════════════════════════════════════════
#  COMPUTER SCIENCE DEPARTMENT
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCS tables")


# ── cs_task_type ───────────────────────────────────────────────────────────────
task_types = []
for type in TASK_TYPES:
    tid    = next_id("cs_task_type")
    task_types.append({
        "task_type_id":   tid,
        "name":           type
    })
write_csv("cs_task_type", task_types)
task_type_ids = [r["task_type_id"] for r in task_types]
# ── cs_algorithm ───────────────────────────────────────────────────────────────
ALGO_NAMES = [
    ("XGBoost 1.7",                  "Gradient Boosting"),
    ("LightGBM 3.3",                 "Gradient Boosting"),
    ("CatBoost 1.2",                 "Gradient Boosting"),
    ("Random Forest (sklearn)",      "Random Forest"),
    ("Extra Trees (sklearn)",        "Random Forest"),
    ("Linear SVM (C=1)",             "SVM"),
    ("RBF SVM (C=10)",               "SVM"),
    ("Logistic Regression (L2)",     "Logistic Regression"),
    ("BERT-base-uncased",            "Transformer"),
    ("RoBERTa-large",                "Transformer"),
    ("DistilBERT",                   "Transformer"),
    ("GPT2-fine-tuned",              "Transformer"),
    ("ResNet-50",                    "Neural Network"),
    ("EfficientNet-B3",              "Neural Network"),
    ("MLP-3layer (ReLU)",            "Neural Network"),
    ("TabNet v2",                    "Neural Network"),
    ("KNN (k=5, cosine)",            "KNN"),
    ("KNN (k=15, euclidean)",        "KNN"),
    ("Decision Tree (max_depth=8)",  "Gradient Boosting"),
    ("AdaBoost (n=100)",             "Gradient Boosting"),
    ("Ridge Regression (α=1.0)",     "Logistic Regression"),
    ("Lasso Regression (α=0.01)",    "Logistic Regression"),
    ("Gaussian Naive Bayes",         "Logistic Regression"),
    ("Isolation Forest (outlier)",   "Neural Network"),
    ("DBSCAN (ε=0.3)",               "KNN"),
]
algorithms = []
algo_family_map = {}
algo_task_map   = {}
for aname, afamily in ALGO_NAMES:
    task = int(np.random.choice(task_type_ids, p=get_powerlaw_p(len(task_type_ids))))
    aid  = next_id("cs_algorithm")
    algorithms.append({
        "algorithm_id": aid,
        "name":         aname,
        "family":       afamily,
        "task_type":    task,
        "paper_doi":    f"10.1000/{fake.bothify('???.####')}",
        "repo_url":     fake.url(),
        "version":      f"{random.randint(1,3)}.{random.randint(0,9)}.{random.randint(0,9)}",
    })
    algo_family_map[aid] = afamily
    algo_task_map[aid]   = task
write_csv("cs_algorithm", algorithms)
algorithm_ids = [r["algorithm_id"] for r in algorithms]

# ── cs_benchmark ───────────────────────────────────────────────────────────────
benchmarks = []
bench_metric_map  = {}
bench_dataset_map = {}
for _ in range(250):
    task   = int(np.random.choice(task_type_ids, p=get_powerlaw_p(len(task_type_ids))))
    metric = random.choice(EVAL_METRICS)
    did    = int(np.random.choice(dataset_ids))
    bid    = next_id("cs_benchmark")
    benchmarks.append({
        "benchmark_id":   bid,
        "name":           f"{fake.word().title()} v{random.randint(1,4)}",
        "task_type":      task,
        "dataset_id":     did,
        "train_split_pct":random.choice([60.0,70.0,75.0,80.0]),
        "eval_metric":    metric,
        "created_at":     rnd_date(date(2020,1,1), date(2024,1,1)),
        "notes":          fake.sentence(),
    })
    bench_metric_map[bid]  = metric
    bench_dataset_map[bid] = did
write_csv("cs_benchmark", benchmarks)
benchmark_ids = [r["benchmark_id"] for r in benchmarks]

# ── cs_experiment_run ──────────────────────────────────────────────────────────
runs = []
for bid in benchmark_ids:
    for _ in range(random.randint(15, 30)):
        aid    = int(np.random.choice(algorithm_ids, p=get_skewed_p(len(algorithm_ids), skew=-4, loc=22, scale=5)))
        status = wc(["Completed","OOM","Timeout"], [88,7,5])
        run_date = rnd_date(date(2021,1,1), date(2024,6,1))
        runs.append({
            "run_id":          next_id("cs_run"),
            "benchmark_id":    bid,
            "algorithm_id":    aid,
            "run_at":          datetime.combine(run_date, datetime.min.time()),
            "hardware_tag":    random.choice(HARDWARE_TAGS),
            "random_seed":     random.randint(0, 99999),
            "train_time_secs": round(random.normalvariate(10, 72000), 3) if status == "Completed" else None,
            "peak_memory_mb":  round(random.uniform(512, 40960), 2) if status == "Completed" else None,
            "git_commit":      fake.sha1()[:40],
            "status":          status,
        })
write_csv("cs_experiment_run", runs)

# ── cs_run_metric ──────────────────────────────────────────────────────────────
run_metrics = []
for run in runs:
    if run["status"] != "Completed":
        continue
    task    = algo_task_map.get(run["algorithm_id"], "Classification")
    metrics = METRICS_BY_TASK.get(task, ["Accuracy","F1"])
    base    = {m: random.gauss(0.55, 0.97)
               if m in ("Accuracy","F1","AUC","Precision","Recall","R2","Silhouette")
               else random.gauss(0.05, 2.5)
               for m in metrics}
    for mname in metrics:
        for split in ["train","val","test"]:
            noise = random.uniform(-0.04, 0.0) if split == "test" else random.uniform(0.0, 0.03)
            run_metrics.append({
                "run_id":      run["run_id"],
                "metric_name": mname,
                "split":       split,
                "value":       round(max(0.0, base[mname] + noise), 6),
                "std_dev":     round(random.uniform(0.001, 0.05), 6),
                "k_folds":     random.choice([None, 5, 10]),
            })
write_csv("cs_run_metric", run_metrics)

# ── cs_hyperparameter ──────────────────────────────────────────────────────────
hyperparams = []
for run in runs:
    if run["status"] != "Completed":
        continue
    family = algo_family_map.get(run["algorithm_id"], "Gradient Boosting")
    hps    = COMMON_HPS.get(family, [("lr","float"),("epochs","int")])
    for pname, ptype in hps:
        hyperparams.append({
            "run_id":      run["run_id"],
            "param_name":  pname,
            "param_value": HP_VALUES[ptype](),
            "param_type":  ptype,
        })
write_csv("cs_hyperparameter", hyperparams)

# ── cs_user_study ──────────────────────────────────────────────────────────────
USER_STUDY_SYSTEMS = [
    "Churn Prediction Dashboard",   "Fraud Alert UI",
    "Recommendation Engine Interface", "AutoML Pipeline Wizard",
    "Loan Risk Scoring Tool",        "NLP Query Interface",
    "Anomaly Heatmap Explorer",      "A/B Test Results Dashboard",
    "Customer Segmentation View",    "Model Explainability Panel",
    "Job Market Trend Visualiser",   "Consumer Sentiment Monitor",
]
user_studies = []
for system in USER_STUDY_SYSTEMS:
    start = rnd_date(date(2021,6,1), date(2023,12,1))
    end   = rnd_date(start + timedelta(days=14), start + timedelta(days=90))
    user_studies.append({
        "user_study_id":  next_id("cs_user_study"),
        "title":          f"User Study: {system}",
        "system_tested":  system,
        "algorithm_id":   int(np.random.choice(algorithm_ids)) if random.random() > 0.3 else None,
        "instrument_id":  int(np.random.choice(instrument_ids)),
        "start_date":     start,
        "end_date":       end,
        "n_participants": random.randint(20, 60),
        "protocol":       random.choice(PROTOCOLS),
    })
write_csv("cs_user_study", user_studies)

# ── cs_study_observation ───────────────────────────────────────────────────────
observations = []
for us in user_studies:
    pool = random.sample(respondent_ids, k=min(us["n_participants"], len(respondent_ids)))
    for rid in pool:
        for task in random.sample(TASK_LABELS, k=random.randint(2, 5)):
            observations.append({
                "obs_id":          next_id("cs_obs"),
                "user_study_id":   us["user_study_id"],
                "respondent_id":   rid,
                "task_label":      task,
                "completion_secs": round(random.uniform(15, 600), 2),
                "error_count":     random.randint(0, 8),
                "nasa_tlx_score":  round(random.uniform(10, 90), 2),
                "satisfaction":    rnd_score(),
                "noted_at":        datetime.combine(
                                       rnd_date(us["start_date"], us["end_date"]),
                                       datetime.min.time()),
            })
write_csv("cs_study_observation", observations)




# =============================================================================
#  JSON-SERVER  --  cs_db.json
# =============================================================================
# json-server expects top-level keys mapping to lists of objects, and works
# best when each object has an "id" field for full REST CRUD support.
# We rename the natural PK column to "id" in the JSON copy only.

PK_FIELDS = {
    "survey_instrument":    "instrument_id",
    "open_dataset":         "dataset_id",
    "respondent":           "respondent_id",
    "cs_algorithm":         "algorithm_id",
    "cs_benchmark":         "benchmark_id",
    "cs_experiment_run":    "run_id",
    "cs_run_metric":        None,
    "cs_hyperparameter":    None,
    "cs_user_study":        "user_study_id",
    "cs_study_observation": "obs_id",
    "cs_task_type": "task_type_id"
}

def _ser(v):
    if isinstance(v, (date, datetime)):
        return v.isoformat()
    return v

def to_collection(rows, pk):
    out = []
    for row in rows:
        r = {k: _ser(v) for k, v in row.items()}
        if pk and pk in r:
            r["id"] = r.pop(pk)
        out.append(r)
    return out

cs_db = {
    # shared tables CS depends on
    "survey_instrument":    to_collection(instruments,   "instrument_id"),
    "open_dataset":         to_collection(datasets,      "dataset_id"),
    "respondent":           to_collection(respondents,   "respondent_id"),
    # CS tables
    "cs_algorithm":         to_collection(algorithms,    "algorithm_id"),
    "cs_benchmark":         to_collection(benchmarks,    "benchmark_id"),
    "cs_experiment_run":    to_collection(runs,          "run_id"),
    "cs_run_metric":        to_collection(run_metrics,   None),
    "cs_hyperparameter":    to_collection(hyperparams,   None),
    "cs_user_study":        to_collection(user_studies,  "user_study_id"),
    "cs_study_observation": to_collection(observations,  "obs_id"),
    "cs_task_type": to_collection(task_types, "task_type_id"),
}

json_path = OUT / "cs_db.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(cs_db, f, indent=2, default=str)

total_json_rows = sum(len(v) for v in cs_db.values())
print(f"  {'cs_db.json':<40} {total_json_rows:>6} rows across {len(cs_db)} collections")
print(f"  Collections: {', '.join(cs_db.keys())}")
print(f"\n  To serve locally:")
print(f"    npx json-server --watch {json_path} --port 3001")

print(f"\nAll output written to {OUT.resolve()}/")


Shared tables
  survey_instrument                            15 rows  →  csv_output/survey_instrument.csv
  open_dataset                                 30 rows  →  csv_output/open_dataset.csv
  respondent                                  800 rows  →  csv_output/respondent.csv

Business tables
  biz_study                                    20 rows  →  csv_output/biz_study.csv
  biz_study_dataset                            47 rows  →  csv_output/biz_study_dataset.csv
  biz_survey_response                        1197 rows  →  csv_output/biz_survey_response.csv
  biz_response_item                         13067 rows  →  csv_output/biz_response_item.csv
  biz_consumer_decision                      1596 rows  →  csv_output/biz_consumer_decision.csv
  biz_employment_wave                        3540 rows  →  csv_output/biz_employment_wave.csv
  biz_market_sentiment                         66 rows  →  csv_output/biz_market_sentiment.csv

CS tables
  cs_task_type                                